# Cable Equation: Spatial Dynamics in Neurons

## Overview
The cable equation describes how voltage propagates along the length of a neuron's dendrite or axon. It's a Partial Differential Equation (PDE) combining temporal and spatial dynamics.

## Mathematical Framework

### Cable Equation (PDE):
$$\frac{\partial V}{\partial t} = \frac{1}{r_a C_m}\frac{\partial^2 V}{\partial x^2} - \frac{1}{C_m}\left[g_L(V - E_L) + I_{\text{ion}}(V,t)\right] + \frac{I_{\text{ext}}(x,t)}{C_m}$$

Simplified (passive cable):
$$\frac{\partial V}{\partial t} = D\frac{\partial^2 V}{\partial x^2} - \frac{V}{\tau} + I_{\text{ext}}(x,t)$$

where:
- $V(x,t)$: membrane voltage at position $x$ and time $t$
- $D$: diffusion coefficient (space constant)
- $\tau$: membrane time constant
- $I_{\text{ext}}(x,t)$: external current input

## Numerical Solution: Crank-Nicolson Method

The Crank-Nicolson method is an **implicit** scheme that's unconditionally stable for parabolic PDEs.

It averages forward and backward Euler:
$$\frac{V^{n+1}_i - V^n_i}{\Delta t} = \frac{1}{2}\left[D\frac{\partial^2 V^{n+1}}{\partial x^2} + D\frac{\partial^2 V^{n}}{\partial x^2}\right] - \frac{1}{2\tau}(V^{n+1}_i + V^n_i) + I$$

This leads to a **tridiagonal linear system** that we solve at each time step.

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
from mpl_toolkits.mplot3d import Axes3D

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 10)

## Model Parameters

In [ ]:
# Physical parameters
L = 1.0           # Length of cable (cm)
T = 50.0          # Total time (ms)
dx = 0.01         # Spatial step (cm)
dt = 0.1          # Time step (ms)

# Cable properties
D = 0.01          # Diffusion coefficient (cm²/ms)
tau = 10.0        # Time constant (ms)

# Discretization
nx = int(L / dx) + 1
nt = int(T / dt) + 1
x = np.linspace(0, L, nx)
t = np.linspace(0, T, nt)

print(f"Cable Equation Parameters:")
print(f"  Cable length: {L} cm")
print(f"  Simulation time: {T} ms")
print(f"  Spatial grid points: {nx}")
print(f"  Temporal grid points: {nt}")
print(f"  Diffusion coefficient: {D} cm²/ms")
print(f"  Time constant: {tau} ms")

## Initialize Solution and Current Injection

In [ ]:
# Initialize voltage array
V = np.zeros((nt, nx))

# External current injection (localized at center, brief pulse)
I_ext = np.zeros((nt, nx))
for n in range(nt):
    for i in range(nx):
        # Current at center (48-52% of cable), from 5-15 ms
        if 0.48 <= x[i] <= 0.52 and 5 <= t[n] <= 15:
            I_ext[n, i] = 10.0

# Visualize current injection
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Spatial profile
ax1.plot(x, I_ext[int(10/dt), :], 'r', linewidth=2)
ax1.set_xlabel('Position x (cm)', fontsize=12)
ax1.set_ylabel('Current (µA/cm²)', fontsize=12)
ax1.set_title('Current Injection: Spatial Profile (t=10 ms)', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Temporal profile at center
center_idx = int(0.5 / dx)
ax2.plot(t, I_ext[:, center_idx], 'b', linewidth=2)
ax2.set_xlabel('Time (ms)', fontsize=12)
ax2.set_ylabel('Current (µA/cm²)', fontsize=12)
ax2.set_title('Current Injection: Temporal Profile (x=0.5 cm)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/cable_current_injection.png', dpi=150, bbox_inches='tight')
plt.show()

## Crank-Nicolson Implementation

Build tridiagonal matrix system and solve at each time step

In [ ]:
# Crank-Nicolson coefficients
r = D * dt / (2 * dx**2)
decay = dt / (2 * tau)

print(f"\nNumerical parameters:")
print(f"  r = D·Δt/(2·Δx²) = {r:.6f}")
print(f"  decay = Δt/(2·τ) = {decay:.6f}")
print(f"  Stability parameter: 2r = {2*r:.6f}")
print(f"  (Crank-Nicolson is unconditionally stable)")

In [ ]:
# Build tridiagonal matrices
# For implicit part (left-hand side): A·V^(n+1) = rhs
main_diag = np.ones(nx) * (1 + 2*r + decay)
off_diag = np.ones(nx-1) * (-r)

# Boundary conditions: sealed ends (no flux)
main_diag[0] = 1 + r + decay
main_diag[-1] = 1 + r + decay

A = diags([off_diag, main_diag, off_diag], [-1, 0, 1], format='csr')

print(f"\nMatrix A (tridiagonal):")
print(f"  Shape: {A.shape}")
print(f"  Non-zero elements: {A.nnz}")

## Time Integration

In [ ]:
print("\nStarting time integration...")

for n in range(nt-1):
    # Right-hand side (explicit part)
    rhs = np.zeros(nx)
    
    # Interior points
    for i in range(1, nx-1):
        rhs[i] = (V[n, i] * (1 - 2*r - decay) + 
                  r * (V[n, i+1] + V[n, i-1]) + 
                  dt * I_ext[n, i])
    
    # Boundary conditions (sealed ends)
    rhs[0] = V[n, 0] * (1 - r - decay) + r * V[n, 1] + dt * I_ext[n, 0]
    rhs[-1] = V[n, -1] * (1 - r - decay) + r * V[n, -2] + dt * I_ext[n, -1]
    
    # Solve linear system: A·V^(n+1) = rhs
    V[n+1, :] = spsolve(A, rhs)
    
    if (n+1) % 100 == 0:
        print(f"  Progress: {(n+1)/nt*100:.1f}%")

print("Integration complete!")
print(f"\nVoltage statistics:")
print(f"  Max voltage: {V.max():.4f} mV")
print(f"  Min voltage: {V.min():.4f} mV")

## Visualization: Spatiotemporal Dynamics

In [ ]:
# Select time snapshots
time_snapshots = [5, 10, 15, 20, 30, 40]
time_indices = [int(t_snap / dt) for t_snap in time_snapshots]

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (t_snap, t_idx) in enumerate(zip(time_snapshots, time_indices)):
    axes[idx].plot(x, V[t_idx, :], 'b', linewidth=2)
    axes[idx].set_xlabel('Position x (cm)', fontsize=11)
    axes[idx].set_ylabel('Voltage V (mV)', fontsize=11)
    axes[idx].set_title(f't = {t_snap} ms', fontsize=12, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylim([V.min()-0.5, V.max()+0.5])
    axes[idx].axvline(0.5, color='r', linestyle='--', alpha=0.3, label='Injection site')
    if idx == 0:
        axes[idx].legend(fontsize=9)

fig.suptitle('Cable Equation: Voltage Propagation Over Time', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/cable_snapshots.png', dpi=150, bbox_inches='tight')
plt.show()

## Space-Time Heatmap

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

# Create meshgrid
X, T_grid = np.meshgrid(x, t)

# Plot heatmap
im = ax.contourf(X, T_grid, V, levels=50, cmap='viridis')
ax.set_xlabel('Position x (cm)', fontsize=12)
ax.set_ylabel('Time (ms)', fontsize=12)
ax.set_title('Spatiotemporal Voltage Dynamics', fontsize=14, fontweight='bold')

# Colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Voltage (mV)', fontsize=12)

# Mark injection region
ax.axvline(0.48, color='red', linestyle='--', alpha=0.5, linewidth=1)
ax.axvline(0.52, color='red', linestyle='--', alpha=0.5, linewidth=1)
ax.axhline(5, color='red', linestyle='--', alpha=0.5, linewidth=1)
ax.axhline(15, color='red', linestyle='--', alpha=0.5, linewidth=1)

plt.tight_layout()
plt.savefig('outputs/cable_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3D Surface Plot

In [ ]:
fig = plt.figure(figsize=(14, 9))
ax = fig.add_subplot(111, projection='3d')

# Subsample for clarity
X_sub, T_sub = np.meshgrid(x[::2], t[::5])
V_sub = V[::5, ::2]

surf = ax.plot_surface(X_sub, T_sub, V_sub, cmap='viridis', 
                        linewidth=0, antialiased=True, alpha=0.9)

ax.set_xlabel('Position x (cm)', fontsize=11)
ax.set_ylabel('Time (ms)', fontsize=11)
ax.set_zlabel('Voltage V (mV)', fontsize=11)
ax.set_title('3D Voltage Landscape', fontsize=14, fontweight='bold')

# Colorbar
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10)

# Adjust viewing angle
ax.view_init(elev=25, azim=45)

plt.tight_layout()
plt.savefig('outputs/cable_3d.png', dpi=150, bbox_inches='tight')
plt.show()

## Analysis: Voltage Decay with Distance

In [ ]:
# Extract voltage at different positions during peak response
peak_time_idx = int(15 / dt)  # Peak around 15 ms
center_idx = int(0.5 / dx)

# Calculate distance from injection site
distance = np.abs(x - 0.5)

# Voltage at peak time
V_peak = V[peak_time_idx, :]

# Theoretical exponential decay: V ~ exp(-x/λ)
# Space constant λ = sqrt(D * tau)
lambda_space = np.sqrt(D * tau)
print(f"\nSpace constant λ = {lambda_space:.4f} cm")

# Fit exponential
V_center = V_peak[center_idx]
V_theory = V_center * np.exp(-distance / lambda_space)

fig, ax = plt.subplots(1, 1, figsize=(12, 6))

ax.plot(distance, V_peak, 'bo', markersize=5, label='Simulated', alpha=0.6)
ax.plot(distance, V_theory, 'r-', linewidth=2, label=f'Theory: V₀·exp(-x/λ), λ={lambda_space:.3f} cm')

ax.set_xlabel('Distance from injection site (cm)', fontsize=12)
ax.set_ylabel('Voltage (mV)', fontsize=12)
ax.set_title('Voltage Decay with Distance (t=15 ms)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

plt.tight_layout()
plt.savefig('outputs/cable_decay.png', dpi=150, bbox_inches='tight')
plt.show()

## Time Course at Different Locations

In [ ]:
# Select positions
positions = [0.5, 0.6, 0.7, 0.8, 0.9]  # cm
position_indices = [int(pos / dx) for pos in positions]

fig, ax = plt.subplots(1, 1, figsize=(12, 6))

colors = plt.cm.plasma(np.linspace(0, 1, len(positions)))

for pos, idx, color in zip(positions, position_indices, colors):
    ax.plot(t, V[:, idx], color=color, linewidth=2, label=f'x = {pos} cm')

ax.set_xlabel('Time (ms)', fontsize=12)
ax.set_ylabel('Voltage (mV)', fontsize=12)
ax.set_title('Voltage Time Course at Different Positions', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)

# Mark stimulation period
ax.axvspan(5, 15, alpha=0.2, color='red', label='Stimulation')

plt.tight_layout()
plt.savefig('outputs/cable_timecourse.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nObservations:")
print("  - Voltage amplitude decreases with distance")
print("  - Voltage peaks later at distant locations (propagation delay)")
print("  - Voltage becomes smoother (filtered) at distant locations")

## Key Takeaways

1. **PDE Structure**: Cable equation combines diffusion (spatial) and decay (temporal)
2. **Crank-Nicolson**: Implicit method, unconditionally stable, 2nd order accurate
3. **Space Constant**: Voltage decays exponentially with distance: λ = √(D·τ)
4. **Propagation Delay**: Signals take time to travel along the cable
5. **Signal Filtering**: High frequencies are attenuated at distant locations

## Biological Implications

- **Dendritic Integration**: Synaptic inputs are filtered and delayed
- **Cable Properties**: Determines effective dendritic tree size
- **Active Propagation**: Real neurons need active conductances (e.g., Na⁺ channels) for long-distance transmission
- **Compartmental Models**: Discretize cable into segments for computational efficiency

## Further Exploration

- Add active conductances (Na⁺, K⁺ channels)
- Implement different boundary conditions (sealed, killed, reflecting)
- Compare explicit vs implicit methods
- Study dendritic branch points
- Implement realistic dendritic morphologies